# MMAR Simulation — Google Colab

[mmar-sim](https://github.com/Rifur/mmar-sim) · notebook **v0.1.2**

執行後會同時顯示：

1. **完整文字報告**（`print`：百分位、機率表、回檔／現價情境、VaR、GOF）
2. **兩張圖表**（路徑圖 + GOF 檢定圖）

順序：先全部文字 → 再圖表。請在 output 區向上捲動閱讀報告。

1. **Runtime → Run all**
2. **Step 2** 改 `TICKER` / `N_STEPS` / `N_SIMS`

範例：`2330.TW`、`6919.TW`、`NVDA`

## Step 1 — Setup

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Rifur/mmar-sim.git"


def _ensure_repo() -> None:
    if os.path.isfile("real_fractal_sim.py"):
        return
    if os.path.isdir("mmar-sim") and os.path.isfile("mmar-sim/real_fractal_sim.py"):
        os.chdir("mmar-sim")
        return
    print("Cloning mmar-sim from GitHub...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL])
    os.chdir("mmar-sim")


_ensure_repo()
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "yfinance>=1.4", "matplotlib>=3.8", "pandas>=2.0",
    "scipy>=1.11", "curl-cffi>=0.15",
])
print(f"Ready: {os.getcwd()}")

## Step 2 — 報告 + 圖表

修改股票代號與參數後執行。`run_colab()` 預設 **文字報告與圖表都會顯示**。

In [ ]:
%matplotlib inline
from real_fractal_sim_colab import run_colab

TICKER = "6919.TW"   # 2330.TW, NVDA, 0050.TW ...
N_STEPS = 20         # 模擬天數（交易日）
N_SIMS = 10000       # 路徑數

result = run_colab(TICKER, n_steps=N_STEPS, n_sims=N_SIMS, seed=42)

## Step 3 — 表單輸入（可選）

輸入股票代號，點 **Run MMAR**。報告與圖表都顯示在下方區域（先文字、後圖）。

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from real_fractal_sim_colab import run_colab

ticker_in = widgets.Text(value="2330.TW", description="Ticker", layout=widgets.Layout(width="280px"))
steps_in = widgets.IntSlider(value=252, min=5, max=504, step=1, description="Days", readout_format="d")
sims_in = widgets.IntSlider(value=5000, min=500, max=20000, step=500, description="Paths", readout_format="d")
run_btn = widgets.Button(description="Run MMAR", button_style="success", icon="play")
report_out = widgets.Output(
    layout=widgets.Layout(width="100%", height="720px", border="1px solid #ccc", overflow_y="auto")
)


def _on_run(_):
    with report_out:
        clear_output(wait=True)
        r = run_colab(
            ticker_in.value.strip().upper(),
            n_steps=steps_in.value,
            n_sims=sims_in.value,
            seed=42,
            show_plots=True,
        )
        globals()["result"] = r


run_btn.on_click(_on_run)
display(widgets.VBox([ticker_in, steps_in, sims_in, run_btn, report_out]))

## 一行程式碼

```python
from real_fractal_sim_colab import run_colab
result = run_colab("6919.TW", n_steps=20, n_sims=10000)
print(result["fit"]["score"])  # 報告與圖表已在上方顯示
```